In [ ]:
#N фиксированные размеры N1=30, N2=

# Описание эксперимента из класса DataSetGenerator:

* N выбирается случайно из диапазонов [20..30] для трейна и [50..100] для теста
* Вариант генерации аттрибутов задает пользователь снаружи (random или identity matrix или assortative) (задать итерирование по разным размерам?)
* Размерность фичей тоже задается пользователем (добавить итерирование по разным размерам?)
* Для assortative варианта sigma_every,sigma_init задает пользователь (добавить рандомную генерацию?) 
* У меня только train test
* только следующие структуры: tree,complete, cycle, regular4, path, ladder (добавить в след раз: general,grid, barbell, expander, ER,BA,SBM) 

In [1]:
from networkx import binomial_tree,complete_graph,cycle_graph,random_regular_graph,path_graph,grid_graph,barbell_graph,ladder_graph# as tree,complete,cycle,regular4,path,grid,barbell,ladder
from torch_geometric.utils.convert import from_networkx 
from networkx.convert_matrix import to_numpy_matrix as adj
import pickle
import os
import numpy as np
import itertools as it
import torch
from torch_geometric.utils.convert import from_networkx,to_scipy_sparse_matrix
import networkx as nx
from functools import reduce

class DataSetGenerator():
    '''
    attr: 'random', 'identity','assortative'
    d: dimensionality of attributes
    '''
    def __init__(self,d=32,attr='random',sigma_init=1,sigma_every=4):
        self.d = d
        if attr =='identity':
            self.attributes = lambda x: torch.eye(x,self.d)
        elif attr == 'random':
            self.attributes = lambda x: torch.normal(torch.zeros(x,self.d),2*torch.ones(x,self.d))
        elif attr== 'assortatve':
            self.attributes = self.generate_attributes
        super().__init__()
    #TODO add sigma_init,sigma_every setup
    
    def graph_generator(self,gen,N):
        
        if gen == random_regular_graph:
            G = gen(4,N)
        else:
            if gen==binomial_tree:
                k=int(np.round(np.log2(N)))
            else:
                k=N
            G = gen(k)
        N=G.number_of_nodes()

        data = from_networkx(G)
        data.x=self.attributes(N)
        xt=(data.x).t()
        data.y =np.sum(torch.mm(data.x,xt)*adj(G))
        return data
    
   # def generate_identity_attributes(self,N,d):
    #    return torch.eye(N,d)
    
   # def generate_random_attributes(self,N,d):
     #   return torch.normal(torch.zeros(N,d),2*torch.ones(N,d))
    
    def generate_attributes(self,G,d,sigma_init,sigma_every):
        partition = community_louvain.best_partition(G)
        len_of_every_partition = {}
        for i in partition:
            if partition[i] not in len_of_every_partition:
                len_of_every_partition[partition[i]] =1
            else:
                len_of_every_partition[partition[i]] +=1      
        Centers = torch.normal(torch.zeros(len(len_of_every_partition),d), torch.ones(len(len_of_every_partition),d)*sigma_init)
        X = torch.zeros(G.number_of_nodes(),d)
        for i in partition:
                X[i]=Centers[partition[i]]+torch.normal(torch.zeros(d),torch.ones(d)*sigma_every)
                    
                    
        return X
    def old_dataset_generator(self):
        all_structures = [binomial_tree,complete_graph, cycle_graph, random_regular_graph, path_graph, ladder_graph]
        
        all_datasets = []
        q=0
        for i in range(1,len(all_structures)):
            train_structures=list(it.combinations(all_structures,i))
            for struct in train_structures:
                struct=list(struct)
                struct.append('')
                struct.reverse()
                
                residuals = list(set(all_structures) - set(struct))
                for j in range(1,len(residuals)+1):
                    test_structures = list(it.combinations(residuals,j))
                    for struct_test in test_structures:
                        all_datasets.append((struct,struct_test))
                        
                        dataset = []
                        
                        for k in range(1000):
                            N_tr = np.random.randint(low=20,high=30) #5000
                            gen = np.random.choice(struct[1:]) 
                            data = self.graph_generator(gen,N_tr)
                            dataset.append(data)
                        for k in range(500):
                            N_te = np.random.randint(low=50,high=100) #2500
                            gen = np.random.choice(struct_test) 
                            data = self.graph_generator(gen,N_te)
                            dataset.append(data)
                        

                        struct_test=list(struct_test)
                        struct_test.append('')
                        struct_test.reverse()
                        
                        #if len(struct)>1:
                        name_tr = reduce(lambda a, b: a+'_'+('_').join(b.__name__.split('_')[:-1])+'_',struct)
                        #else:
                        #    name_tr = ('_').join(struct[0].__name__.split('_')[:-1])
                        #if len(struct_test)>1:
                        name_te = reduce(lambda a, b: a+'_'+('_').join(b.__name__.split('_')[:-1])+'_',struct_test)+'.pickle'
                        #else:
                         #   name_te = ('_').join(struct_test[0].__name__.split('_')[:-1])+'.pickle'
                        if not os.path.exists('datasets/'+name_tr+'_test_'+name_te):
                            with open('datasets/'+name_tr+'_test_'+name_te,'wb') as f:
                                pickle.dump(dataset,f)
                        print(q)
                        q+=1
        
    def run(self):           
            all_structures = [binomial_tree,complete_graph, cycle_graph, random_regular_graph, path_graph, ladder_graph]
            
            for struct in all_structures:
                print(struct.__name__)
                dataset_train = []
                dataset_test = [] 
                for i in range(5000):
                    N_tr = np.random.randint(low=20,high=30) #5000
                    data = self.graph_generator(struct,N_tr)
                    dataset_train.append(data)
                for k in range(2500):
                    N_te = np.random.randint(low=50,high=100) #2500
                    data = self.graph_generator(struct,N_te)
                    dataset_test.append(data)
                
                name_tr = struct.__name__.split('_')[0]+'_train.pickle'
                        #else:
                        #    name_tr = ('_').join(struct[0].__name__.split('_')[:-1])
                        #if len(struct_test)>1:
                name_te = struct.__name__.split('_')[0]+'_test.pickle'
                        #else:
                         #   name_te = ('_').join(struct_test[0].__name__.split('_')[:-1])+'.pickle'
                
                if not os.path.exists('datasets/'+name_tr):
                    with open('datasets/'+name_tr,'wb') as f:
                                pickle.dump(dataset_train,f)
                if not os.path.exists('datasets/'+name_te):
                    with open('datasets/'+name_te,'wb') as f:
                                pickle.dump(dataset_test,f)
                            

In [2]:
DSGen=DataSetGenerator()
DSGen.run()

binomial_tree
complete_graph
cycle_graph
random_regular_graph
path_graph
ladder_graph


# МБ пригодится

In [3]:
def tree(N,d): #мб взять другой вид дерева их много, вопрос как параметры высчитывать
    G = nx.binomial_tree(int(np.round(np.log2(N))))
    data = from_networkx(G)
    data.x = generate_random_attributes(N,d)
    data.y=3
    return data

def complete(N,d):
    G = nx.complete_graph(N)
    data = from_networkx(G)
    data.x = generate_random_attributes(N,d)
    return data

def cycle(N,d):
    G = nx.cycle_graph(N)
    data = from_networkx(G)
    data.x = generate_random_attributes(N,d)
    return data


def regular4(N,d):
    N=4,N
    print(N)
    G = nx.random_regular_graph(N)
    data = from_networkx(G)
    #data.x = generate_random_attributes(N,d)
    return data

def path(N,d):
    G = nx.path_graph(N)
    data = from_networkx(G)
    data.x = generate_random_attributes(N,d)
    return data
    
def grid(N,d): #как подобрать размер решетки 
    G = nx.grid_graph(6,4)
    data = from_networkx(G)
    data.x = generate_random_attributes(N,d)
    return data

def barbell(N,d): #как подбирать размеры m1,m2
    G = nx.barbell_graph(N)
    data = from_networkx(G)
    data.x = generate_random_attributes(N,d)
    return data

def ladder(N,d):
    G = nx.ladder_graph(N)
    data = from_networkx(G)
    #data.x = generate_random_attributes(N,d)
    
    return data

In [4]:
all_structures = ['tree','complete', 'cycle', 'regular', 'path', 'ladder']
all_datasets=[]
for i in range(1,len(all_structures)):
            train_structures=list(it.combinations(all_structures,i))
            for structure in train_structures:
                residuals = list(set(all_structures) - set(structure))
                for j in range(1,len(residuals)+1):
                    test_structures = list(it.combinations(residuals,j))
                    for struct_test in test_structures:
                        all_datasets.append((struct,struct_test))
len(all_datasets)

NameError: name 'struct' is not defined